In [ ]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import platform
from pprint import pprint

# 전처리
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight # 샘플 가중치: 모델 학습 시 각 샘플에 중요도를 다르게 부여하는 방법

# 모델
from lightgbm import LGBMRegressor                  # LightGBM 회귀모델
from sklearn.ensemble import RandomForestRegressor  # 랜덤 포레스트 회귀모델
from xgboost import XGBRegressor                    # XGBoost 회귀모델

# 모델 학습 및 평가 지표
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error, 
    r2_score
)

# SHAP
import shap

# 모델 저장/불러오기
import joblib
import os

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

# matplotlib 설정
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 고정
np.random.seed(42)

print("=" * 60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("=" * 60)

---
### 1. 데이터로드
**영상 데이터**
- IT/FnB 숏폼 영상 데이터가 하나의 파일에 통합되어 있음
- 도메인별로 100개씩, 총 200개의 영상 데이터가 포함되어 있음

**댓글 데이터**
- IT/FnB 성공/실패 댓글 데이터 각각 로드
- 샘플링된 영상에 해당하는 댓글만 사용

In [ ]:
# 숏츠 영상 분석 데이터
df_video_short = pd.read_csv("../../../data/results/main_dataset/쇼츠 영상(+이미지 분석 정보)/result_sample_shorts_all_for_video_agent_fixed.csv", encoding='utf-8')

# IT 성공 숏츠 댓글
df_success_IT_short_comment = pd.read_csv("../../../data/results/main_dataset/댓글(필터링+감성분석 정보)/sentiment_filtered_cleaned_it_shorts_success_comment.csv", encoding='utf-8')

# IT 실패 숏츠 댓글
df_fail_IT_short_comment = pd.read_csv("../../../data/results/main_dataset/댓글(필터링+감성분석 정보)/sentiment_filtered_cleaned_it_shorts_fail_comment.csv", encoding='utf-8')

# FnB 성공 숏츠 댓글
df_success_FnB_short_comment = pd.read_csv("../../../data/results/main_dataset/댓글(필터링+감성분석 정보)/sentiment_filtered_cleaned_fnb_shorts_success_comment.csv", encoding='utf-8')

# FnB 실패 숏츠 댓글
df_fail_FnB_short_comment = pd.read_csv("../../../data/results/main_dataset/댓글(필터링+감성분석 정보)/sentiment_filtered_cleaned_fnb_shorts_fail_comment.csv", encoding='utf-8')

In [ ]:
# 데이터 확인
print(f"숏폼 영상 데이터: {len(df_video_short)}개")
print(f"\n[IT 댓글]")
print(f"성공: {len(df_success_IT_short_comment)}개")
print(f"실패: {len(df_fail_IT_short_comment)}개")
print(f"\n[FnB 댓글]")
print(f"성공: {len(df_success_FnB_short_comment)}개")
print(f"실패: {len(df_fail_FnB_short_comment)}개")

In [ ]:
# 컬럼명 확인
print("[숏폼 영상 데이터의 컬럼명]")
pprint(df_video_short.columns)

print("\n[댓글데이터의 컬럼명]")
pprint(df_success_IT_short_comment.columns)

---
### 2. 데이터 전처리
#### 2-1. 댓글 데이터 통합
- IT/FnB 성공/실패 댓글 데이터 통합 → 타깃 변수 계산용 임시 데이터 1개

#### 2-2. 댓글 필터링
- 이벤트 참여 댓글(`is_event=True`) 제거
- 외국어 댓글(`is_korean=False`) 제거

#### 2-3. 타깃 변수 생성
- 필터링된 댓글을 `video_id` 기준으로 그룹화
- 영상별 긍정 댓글 비율 계산 → 타깃 변수(`target_feature`) 생성
  - 공식: `긍정 댓글 수 / 전체 댓글 수`
  - 댓글이 없는 영상은 타깃 변수 계산 불가 → 이후 merge 시 제외

#### 2-4. 영상 데이터와 타깃 변수 merge
- 영상 데이터와 타깃 변수를 `video_id` 기준으로 결합
- `how='inner'`로 댓글이 없는 영상은 자동 제외

#### 2-5. feature 선택
- 식별자(`video_id`, `channel_id` 등) 제외
- 성과 지표(`조회수`, `좋아요수`, `score1`, `score2`, `grade` 등) 제외
  - 영상 제작 시점에 알 수 없는 값
  - 타깃 변수와 연관될 수 있어 데이터 누수 위험
- 텍스트 컬럼(`description`, `tags`, `reason` 등) 제외
- 이미지 분석 수치 컬럼(`avg_brightness`, `avg_blue`, `avg_green`, `avg_red`, `text_ratio`) 제외
  - 영상 제작 시점에 입력하기 어려운 값
- 최종 feature 목록 (20개):
  - `domain`, `영상길이(초)`, `has_paid_product_placement`, `channel_tier`
  - `upload_month`, `upload_dayofweek`, `upload_hour`, `upload_quarter`
  - `description_length`, `tags_count`, `production_quality`, `lighting_style`
  - `color_mood`, `editing_pace`, `motion_graphic`, `video_format`
  - `first_3sec`, `background_style`, `person_ratio`, `face_ratio`

#### 2-6. 결측값 처리
- 결측값 현황 확인 후 처리 방법 결정

#### 2-7. 범주형 변수 인코딩
- Label Encoding 적용 (str → int)
  - `domain`, `channel_tier`, `upload_dayofweek`
  - `production_quality`, `lighting_style`, `color_mood`, `editing_pace`
  - `motion_graphic`, `video_format`, `first_3sec`, `background_style`
- bool → int 변환
  - `has_paid_product_placement`

#### 2-8. 데이터 불균형 보정
- 타깃 변수(긍정 댓글 비율) 분포 확인
- 구간화 기반 샘플 가중치 적용
- 보정 방법은 데이터 분포 확인 후 최종 결정

In [ ]:
# ========================================
# [전처리 1] 댓글 데이터 통합
# ========================================

# [역할] IT/FnB 성공/실패 댓글 데이터 통합
#       - 타깃 변수 계산을 위한 임시 댓글 데이터 제작

df_comment_short = pd.concat([
    df_success_IT_short_comment,
    df_fail_IT_short_comment,
    df_success_FnB_short_comment,
    df_fail_FnB_short_comment
], ignore_index=True)

print(f"영상 데이터: {len(df_video_short)}개")
print(f"댓글 데이터: {len(df_comment_short)}개")

In [ ]:
# ========================================
# [전처리 2] 댓글 필터링
# ========================================

# [역할] 이벤트 참여 댓글 및 외국어 댓글 제거

before = len(df_comment_short)
df_comment_short = df_comment_short[df_comment_short['is_event'] == False]
print(f"이벤트 댓글 제거: {before - len(df_comment_short)}개 제거 → {len(df_comment_short)}개 남음")

before = len(df_comment_short)
df_comment_short = df_comment_short[df_comment_short['is_korean'] == True].reset_index(drop=True)
print(f"외국어 댓글 제거: {before - len(df_comment_short)}개 제거 → {len(df_comment_short)}개 남음")

In [ ]:
# ========================================
# [전처리 2-1] 샘플링된 영상 댓글만 필터링
# ========================================

# [역할] df_video_short에 있는 video_id에 해당하는 댓글만 남김
#        댓글 데이터에 샘플링된 200개 영상 외의 댓글이 포함되어 있을 수 있음

valid_video_ids = set(df_video_short['video_id'])

before = len(df_comment_short)
df_comment_short = df_comment_short[
    df_comment_short['video_id'].isin(valid_video_ids)
].reset_index(drop=True)

print(f"샘플링된 영상 댓글만 필터링: {before - len(df_comment_short)}개 제거 → {len(df_comment_short)}개 남음")

In [ ]:
# ========================================
# [전처리 3] 타깃 변수 생성
# ========================================

# [역할] 영상별 긍정 댓글 비율 계산 → 타깃 변수(target_feature) 생성
#        공식: 긍정 댓글 수 / 전체 댓글 수

agg_short = (
    df_comment_short.groupby('video_id')['sentiment']
    .agg(
        total='count',
        positive=lambda x: (x == '긍정').sum()
    )
    .reset_index()
)
agg_short['target_feature'] = agg_short['positive'] / agg_short['total']

print(f"타깃 변수 계산 완료: {len(agg_short)}개 영상")
print(f"\n[긍정 댓글 비율 분포]")
print(agg_short['target_feature'].describe())